# Numerical Fragility MLOps GPU Run

This notebook runs the full experiment matrix on a GPU runtime, regenerates the comparison artifacts, and packages the outputs for download.
Because the Colab runtime has both CPU and CUDA available, this version is intended to produce a real CPU-vs-CUDA `device_sweep` in addition to the CUDA precision comparison.

Before running:
1. In Colab, go to `Runtime -> Change runtime type`.
2. Set `Hardware accelerator` to `GPU`.
3. Run cells top to bottom.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # update this
BRANCH = "main"  # update if needed
REPO_DIR = "/content/numerical-fragility-mlops"

!rm -rf $REPO_DIR
!git clone --branch $BRANCH $REPO_URL $REPO_DIR

In [ ]:
%cd /content/numerical-fragility-mlops
!python3 -m pip install -q --upgrade pip
!python3 -m pip install -q -r requirements.txt

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU runtime is not enabled in Colab.")

In [ ]:
%cd /content/numerical-fragility-mlops
!RUN_MODE=full python3 src/train.py
!python3 src/plot_results.py
!python3 src/generate_results_table.py

In [ ]:
!find artifacts -maxdepth 2 -type f | sort | sed -n '1,200p'

In [ ]:
%cd /content/numerical-fragility-mlops
!rm -f gpu_artifacts_bundle.zip
!zip -r gpu_artifacts_bundle.zip \
    artifacts/comparisons_week4.csv \
    artifacts/predictions \
    artifacts/results_summary_table.csv \
    artifacts/results_summary_table.md \
    artifacts/results_detail_table.csv \
    artifacts/results_detail_table.md \
    artifacts/preprocessing_disagreement.png \
    artifacts/device_disagreement.png \
    artifacts/precision_disagreement.png

In [ ]:
from google.colab import files

files.download("/content/numerical-fragility-mlops/gpu_artifacts_bundle.zip")

## After download

Bring the files from `gpu_artifacts_bundle.zip` back into your local repo, then rerun:

```bash
python3 src/plot_results.py
python3 src/generate_results_table.py
```

That will refresh the repo outputs and give you real CPU-vs-CUDA and CUDA precision artifacts for the reports.